# AudioGen LoRA Fine-Tuning for Minecraft Sound Effects

**Goal:** Fine-tune Meta's AudioGen (autoregressive transformer on EnCodec tokens) with LoRA
to generate Minecraft-style sound effects from text prompts.

**Runtime:** Local GPU (RTX 5090 32 GB) or Google Colab T4

### Pipeline
1. Install dependencies (AudioCraft + EnCodec)
2. Verify data is in place (or fetch & preprocess)
3. Prepare AudioGen dataset (JSONL manifests + JSON sidecars)
4. Generate **baseline** samples from vanilla AudioGen
5. LoRA fine-tune AudioGen's language model
6. Generate **adapted** samples and compare

### Setup
- Clone the repo and `cd` into it
- Copy `data/` folder (processed .wav + manifest.csv) from your other machine
- Open this notebook in VS Code or Jupyter

---
## 0 · Check GPU & Verify Working Directory

In [ ]:
import os
from pathlib import Path

# Ensure we're in the repo root
# Adjust this if your repo is cloned elsewhere
REPO_DIR = Path(os.getcwd())
if not (REPO_DIR / "configs" / "demo1.yaml").exists():
    # Try going up one level (if running from notebooks/)
    REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Verify GPU
import torch
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — training will be very slow")

---
## 1 · Install Dependencies

In [ ]:
# ── Install Python dependencies ──
# If you have a fresh environment, install everything:
!pip install -q librosa soundfile pydub pyyaml requests tqdm scipy
!pip install -q audiocraft encodec

# Verify imports
import torch
print(f"torch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

from audiocraft.models import AudioGen
print("AudioCraft imported successfully")

---
## 2 · Verify Data (or Fetch & Preprocess)

If you copied `data/processed/` and `data/manifest.csv` from another machine, **skip the fetch/preprocess cells** and just run the verification cell.

In [ ]:
# ── SKIP THIS CELL if you already have data/processed/ and data/manifest.csv ──
# Download + preprocess from scratch

!python scripts/fetch_minecraft_assets.py --config configs/demo1.yaml
!python scripts/preprocess_audio.py --config configs/demo1.yaml
!python scripts/build_manifest.py --config configs/demo1.yaml

In [ ]:
# Verify data is in place
import glob
import pandas as pd

wav_files = glob.glob("data/processed/**/*.wav", recursive=True)
print(f"Processed .wav files found: {len(wav_files)}")

assert len(wav_files) > 0, "No .wav files found — run the fetch/preprocess cell above, or copy data/ from another machine"

df = pd.read_csv("data/manifest.csv")
print(f"Manifest: {len(df)} clips  |  train={len(df[df.split=='train'])}  val={len(df[df.split=='val'])}")
df.sample(5)

---
## 3 · Prepare AudioGen Dataset Format

In [ ]:
# Convert manifest → AudioCraft format (JSONL + JSON sidecars)
!python scripts/prepare_audiogen_data.py --config configs/demo1.yaml

# Verify
import json
for split in ["train", "val"]:
    with open(f"data/audiogen/{split}.jsonl") as f:
        entries = [json.loads(l) for l in f]
    print(f"{split}: {len(entries)} entries")
    if entries:
        print(f"  sample: {entries[0]}")

In [ ]:
# ── Sanity check: EnCodec roundtrip ──
# Encode a Minecraft sound → decode → verify it's recognizable
import torch
import soundfile as sf
from audiocraft.models import AudioGen

model = AudioGen.get_pretrained('facebook/audiogen-medium')
encodec = model.compression_model
encodec.eval()

# Pick a sample file
sample_wav = df.iloc[0]['file_name']
audio, sr = sf.read(f"data/processed/{sample_wav}", dtype='float32')
print(f"Original: {sample_wav}  |  sr={sr}  dur={len(audio)/sr:.2f}s")

# Encode → decode
wav_tensor = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0)  # (1, 1, T)
device = next(encodec.parameters()).device
wav_tensor = wav_tensor.to(device)

with torch.no_grad():
    encoded = encodec.encode(wav_tensor)
    codes = encoded[0][0]  # (1, K, T_codes)
    print(f"EnCodec codes shape: {codes.shape}  (K={codes.shape[1]} codebooks, T={codes.shape[2]} frames)")
    decoded = encodec.decode(encoded[0])

decoded_np = decoded[0].cpu().squeeze().numpy()
print(f"Decoded length: {len(decoded_np)/sr:.2f}s")

# Listen: original vs roundtrip
from IPython.display import Audio, display
print("Original:")
display(Audio(audio, rate=sr))
print("EnCodec roundtrip:")
display(Audio(decoded_np, rate=sr))

# Cleanup to free VRAM
del model, encodec, wav_tensor, encoded, decoded
torch.cuda.empty_cache()

---
## 4 · Baseline Generation (Vanilla AudioGen)

In [ ]:
# Generate baseline samples from vanilla AudioGen (no fine-tuning)
BASELINE_PROMPTS = [
    "minecraft zombie getting hurt sound effect",
    "minecraft skeleton death sound effect",
    "minecraft cave ambience sound effect",
    "minecraft walking footsteps on stone surface sound effect",
]

for prompt in BASELINE_PROMPTS:
    !python -m src.mcaudio.infer.audiogen_generate \
        --prompt "{prompt}" \
        --config configs/demo1.yaml \
        --num_samples 2 \
        --output outputs/audiogen/baseline

In [ ]:
# Listen to baseline samples
import glob
import soundfile as sf
from IPython.display import Audio, display

baseline_files = sorted(glob.glob("outputs/audiogen/baseline/*.wav"))[:8]
for f in baseline_files:
    audio, sr = sf.read(f, dtype='float32')
    print(f"\n{os.path.basename(f)}")
    display(Audio(audio, rate=sr))

---
## 5 · LoRA Fine-Tuning

In [ ]:
# ── LoRA fine-tune AudioGen's language model ──
# Trains only LoRA adapters (~44M params) on the frozen 1.5B model.
# On RTX 5090 (32GB): ~30-60 min for 150 epochs.
# On T4 (16GB): ~1-2 hours for 150 epochs.
#
# Can increase --batch_size to 4 or 8 on 5090 for faster training.

!python -m src.mcaudio.train.audiogen_lora_train \
    --config configs/demo1.yaml \
    --epochs 150 \
    --batch_size 4

In [ ]:
# ── Quick smoke test (10 epochs) ──
# Run this FIRST to verify everything works before full training

# !python -m src.mcaudio.train.audiogen_lora_train \
#     --config configs/demo1.yaml \
#     --epochs 10 \
#     --batch_size 4

---
## 6 · Generate with Fine-Tuned Model

In [ ]:
# Generate samples using the best LoRA checkpoint
EVAL_PROMPTS = [
    "minecraft zombie getting hurt sound effect",
    "minecraft skeleton death sound effect",
    "minecraft cave ambience sound effect",
    "minecraft walking footsteps on stone surface sound effect",
]

for prompt in EVAL_PROMPTS:
    !python -m src.mcaudio.infer.audiogen_generate \
        --prompt "{prompt}" \
        --config configs/demo1.yaml \
        --lora_weights outputs/audiogen/lora_weights/best \
        --num_samples 2 \
        --output outputs/audiogen/lora

---
## 7 · Side-by-Side Comparison

In [ ]:
# Compare baseline vs fine-tuned outputs
import glob
import os
import soundfile as sf
from IPython.display import Audio, display, HTML

baseline_dir = "outputs/audiogen/baseline"
lora_dir = "outputs/audiogen/lora"

baseline_files = sorted(glob.glob(f"{baseline_dir}/*.wav"))
lora_files = sorted(glob.glob(f"{lora_dir}/*.wav"))

# Match files by name prefix
for bf in baseline_files[:8]:
    name = os.path.basename(bf)
    lf = os.path.join(lora_dir, name)
    if not os.path.exists(lf):
        continue

    b_audio, sr = sf.read(bf, dtype='float32')
    l_audio, _ = sf.read(lf, dtype='float32')

    display(HTML(f"<h4>{name}</h4>"))
    print("  Baseline (vanilla AudioGen):")
    display(Audio(b_audio, rate=sr))
    print("  Fine-tuned (LoRA):")
    display(Audio(l_audio, rate=sr))

---
## 8 · Compare with Real Minecraft Sounds

In [ ]:
# Listen to a few real Minecraft sounds for reference
import soundfile as sf
from IPython.display import Audio, display

real_samples = [
    ("data/processed/mob/zombie/hurt_seq.wav", "Real: zombie hurt"),
    ("data/processed/mob/skeleton/hurt_seq.wav", "Real: skeleton hurt"),
    ("data/processed/ambient/cave/cave1.wav", "Real: cave ambience"),
    ("data/processed/step/stone_walk.wav", "Real: stone footsteps"),
]

for path, label in real_samples:
    if os.path.exists(path):
        audio, sr = sf.read(path, dtype='float32')
        print(f"\n{label}")
        display(Audio(audio, rate=sr))

---
## Notes

- **FP32 training**: We avoid mixed precision due to a known AudioCraft NaN bug with autocast.
- **LoRA targets**: `out_proj`, `linear1`, `linear2` in the transformer (NOT q/k/v — they're fused).
- **NaN handling**: The delay codebook pattern produces NaN logits at offset positions — `nan_to_num` is critical.
- **CFG dropout**: 10% of captions are dropped during training for classifier-free guidance at inference.
- If training loss doesn't decrease after 5 epochs, try reducing `learning_rate` to `1e-4`.